# Fase 3.7 — expansão interna, execução manual no Colab

A 3.7 acrescenta amostras de dois novos shards do Community Forensics–Small aos dados da 3.5. O dataset continua o mesmo: esta versão testa maior cobertura interna, sem afirmar independência de fonte. Arquitetura, checkpoint inicial da 3.3, aumentos, amostrador e limite de 12 épocas permanecem iguais aos da 3.5. Os diretórios 3.5 e 3.6 são preservados.

Execute as células 1–5 para preparar o runtime. A 6 faz o piloto, que confere classes e grupos, mas não seleciona dados. A 7 baixa e congela a seleção e as divisões. A 8 copia os dados para /content. A 9 é o smoke test: verifica hashes, checkpoint, forward e gravação, sem treinar. A 10 inicia ou retoma o treino manualmente. A 11 lê os artefatos internos. Após o treino, a 12 copia Mirage, a 13 congela checkpoint, temperatura e limiar, a 14 avalia e a 15 lê o resultado. Mirage já foi observado: serve somente para comparação, sem orientar mudanças nesta versão.

Conecte uma GPU. Não use “Run All”: o treino completo ocorre somente na célula 10.


In [ ]:
# 1. GPU e dependências do runtime.
import torch, sys, subprocess
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab antes de continuar'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'huggingface_hub', 'pyarrow', 'scikit-learn'])
print('Dependências disponíveis.', flush=True)

In [ ]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_fase3_7', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

In [ ]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

In [ ]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase37-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/fase3_7_dados.py', 'experimentos/fase3_7_finetune.py', 'experimentos/avaliar_fase3_7_mirage.py', 'Notebooks/fase_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc37_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivos da Fase 3.7 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

In [ ]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
DATA = CODE_ROOT / 'experimentos/fase3_7_dados.py'
TRAIN = CODE_ROOT / 'experimentos/fase3_7_finetune.py'
for path in (DATA, TRAIN):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

In [ ]:
# 6. PILOTO MANUAL: baixa dois shards completos em /content e confere classes e grupos.
SHARDS = ['data/HFCF_small_1.parquet', 'data/HFCF_small_118.parquet']
shard_args = [item for shard in SHARDS for item in ('--shard', shard)]
run_live('piloto 3.7', [sys.executable, '-u', str(DATA), 'pilot', '--root', str(ROOT), *shard_args])


In [ ]:
# 7. PREPARO MANUAL: congela selecao e divisoes antes de qualquer treino.
# Se o piloto indicar classes insuficientes, pare e informe a saida.
run_live('preparacao 3.7', [sys.executable, '-u', str(DATA), 'prepare', '--root', str(ROOT), *shard_args])


In [ ]:
# 8. COPIA LOCAL: execute em cada sessao e confira hashes de ambos os conjuntos.
OLD_DATA = CODE_ROOT / 'experimentos/fase3_5_dados.py'
run_live('copia local 3.5', [sys.executable, '-u', str(OLD_DATA), 'stage', '--root', str(ROOT)])
run_live('copia local 3.7', [sys.executable, '-u', str(DATA), 'stage', '--root', str(ROOT)])


In [ ]:
# 9. SMOKE MANUAL: divisoes, GPU, checkpoint, forward e gravacao; sem treino.
run_live('smoke 3.7', [sys.executable, '-u', str(TRAIN), 'smoke'])


In [ ]:
# 10. TREINO MANUAL: inicia ou retoma do ultimo last.pt.
# Execute somente apos conferir Smoke OK na celula 9.
run_live('ajuste fino 3.7', [sys.executable, '-u', str(TRAIN), 'auto'])


In [ ]:
# 11. LEITURA INTERNA: selecao escolhe epoca; calibracao define T e limiar.
run_dir = ROOT / 'modelos/fase3_7_cfsmall_expansao_v1'
for name in ('experiment_config.json', 'split_manifest.json', 'history.csv', 'last.pt',
             'best.pt', 'phase3_final.pt', 'internal_metrics.json', 'training_complete.json'):
    print(name, 'presente' if (run_dir / name).is_file() else 'AUSENTE')
metrics = run_dir / 'internal_metrics.json'
print(json.dumps(json.loads(metrics.read_text(encoding='utf-8')), indent=2)
      if metrics.is_file() else 'Treino interno ainda nao concluido.')


## Comparação externa após o treino

As células 12–14 executam o protocolo Mirage de 2.000 imagens congeladas. A célula 13 registra hashes de pesos, código e manifesto antes de inferir. Temperatura e limiar vêm da calibração interna. O Mirage já foi observado e não é um teste cego independente para esta variante. Uma futura avaliação final independente exige outro conjunto ainda não observado.


In [ ]:
# 12. MIRAGE: copia local verificada, sem novo download da fonte.
MIRAGE_DATA = CODE_ROOT / 'experimentos/benchmark_mirage_2000.py'
run_live('copia local Mirage', [sys.executable, '-u', str(MIRAGE_DATA), 'stage', '--root', str(ROOT)])


In [ ]:
# 13. CONGELAMENTO: checkpoint, temperatura, limiar, codigo e manifesto.
EVAL = CODE_ROOT / 'experimentos/avaliar_fase3_7_mirage.py'
run_live('congelamento 3.7', [sys.executable, '-u', str(EVAL), 'freeze', '--root', str(ROOT)])


In [ ]:
# 14. AVALIACAO EXTERNA COMPARATIVA apos o congelamento.
run_live('avaliacao Mirage 3.7', [sys.executable, '-u', str(EVAL), 'evaluate', '--root', str(ROOT)])


In [ ]:
# 15. LEITURA do resultado preservado; sem recalibracao.
result = ROOT / 'resultados/benchmark_externo_mirage_2000_v1/fase3_7_cfsmall_expansao_v1/metrics.json'
print(json.dumps(json.loads(result.read_text(encoding='utf-8')), indent=2)
      if result.is_file() else 'Avaliacao externa ainda nao executada.')
